# 1. Load Processed Datasets

This section loads the cleaned supervised dataset and the
unlabelled prediction pool generated in Notebook 04.

The supervised dataset will be used for model development,
while the prediction pool will remain separate for future
batch predictions.

In [1]:
# Import libraries required for tokenization and data preparation
import pandas as pd
import numpy as np

# Import tools for splitting the supervised dataset
from sklearn.model_selection import train_test_split

# Load the cleaned supervised dataset
supervised_df = pd.read_csv(
    "../data/interim/civicsense_supervised_cleaned.csv"
)

# Load the cleaned prediction pool
prediction_df = pd.read_csv(
    "../data/interim/civicsense_prediction_cleaned.csv"
)

# Display the dimensions of both datasets
print("Supervised Dataset Shape:", supervised_df.shape)
print("Prediction Pool Shape:", prediction_df.shape)

# Display sample records from the supervised dataset
display(supervised_df.head())

# Display sample records from the prediction pool
display(
    prediction_df[
        ["cleaned_text", "detectedLanguage"]
    ].head()
)

Supervised Dataset Shape: (16519, 3)
Prediction Pool Shape: (27686, 14)


,cleaned_text,target_category,source_dataset
0,Sadak paar karne wala concrete ka puliya adhur...,Road Infrastructure,Dataset_A_Multilingual
1,"सीएमसी का पानी जारी नहीं किया जा रहा है, वे बा...",Water Supply,Dataset_A_Multilingual
2,Garbage dump all over,Waste Management,Dataset_A_Multilingual
3,Puttenahalli jheel road jo Arekere ki or jaati...,Road Infrastructure,Dataset_A_Multilingual
4,Desilted Mud is not cleared making the road di...,Waste Management,Dataset_A_Multilingual


,cleaned_text,detectedLanguage
0,"Sir, we are facing lot of water problem in 8th...",English
1,"Doddabommasandra Arch se Nanjappa Circle, Nanj...",Hinglish
2,Kachra kachra kachra,Hinglish
3,Garbage dumped next to mysore circle bus stop....,English
4,Too many pot holes near GIIS school and petrol...,English


In [2]:
# Check missing values in the supervised dataset
print("Supervised Dataset Missing Values:")
print(supervised_df.isnull().sum())

# Check missing values in the prediction pool
print("\nPrediction Pool Missing Values:")
print(prediction_df["cleaned_text"].isnull().sum())

# Check the number of target categories
print("\nNumber of Target Categories:",
      supervised_df["target_category"].nunique())

# Display category names
print("\nTarget Categories:")
print(sorted(supervised_df["target_category"].unique()))

# Check the number of records contributed by each source
print("\nSource Dataset Distribution:")
print(supervised_df["source_dataset"].value_counts())

Supervised Dataset Missing Values:
cleaned_text       0
target_category    0
source_dataset     0
dtype: int64

Prediction Pool Missing Values:
0

Number of Target Categories: 12

Target Categories:
['Animal Management', 'Civic Infrastructure and Encroachment', 'Drainage and Sewerage', 'Electricity and Utilities', 'Environment and Public Spaces', 'Public Safety', 'Public Sanitation', 'Road Infrastructure', 'Street Lighting', 'Traffic and Transport', 'Waste Management', 'Water Supply']

Source Dataset Distribution:
source_dataset
Dataset_B_English         15296
Dataset_A_Multilingual     1223
Name: count, dtype: int64


# 2. Prepare Training, Validation and Test Sets

Split the labelled dataset into training, validation and test sets.

Stratified sampling is used to maintain the approximate distribution
of all 12 target categories across the three subsets.

The prediction pool remains separate and will not participate
in model training or evaluation.

In [4]:
# Extract cleaned complaint texts as input features
X = supervised_df["cleaned_text"]

# Extract target categories as classification labels
y = supervised_df["target_category"]

# Display the original dataset dimensions
print("Features Shape:", X.shape)
print("Labels Shape:", y.shape)

# Display category distribution before splitting
print("\nOriginal Category Distribution:")
print(y.value_counts())

Features Shape: (16519,)
Labels Shape: (16519,)

Original Category Distribution:
target_category
Road Infrastructure                      5224
Waste Management                         4235
Street Lighting                          1472
Traffic and Transport                    1159
Drainage and Sewerage                    1006
Animal Management                         868
Environment and Public Spaces             758
Water Supply                              602
Public Sanitation                         481
Electricity and Utilities                 269
Civic Infrastructure and Encroachment     242
Public Safety                             203
Name: count, dtype: int64


In [5]:
# Split 15% of the data for the final test set
# Stratification preserves the approximate category proportions
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

# Split the remaining 85% into training and validation sets
# 15% of the complete dataset is reserved for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.15 / 0.85,
    random_state=42,
    stratify=y_train_val
)

# Display the dimensions of all three subsets
print("Training Set:", X_train.shape, y_train.shape)
print("Validation Set:", X_val.shape, y_val.shape)
print("Test Set:", X_test.shape, y_test.shape)

# Verify that all subsets contain the 12 target categories
print("\nUnique categories:")
print("Training:", y_train.nunique())
print("Validation:", y_val.nunique())
print("Test:", y_test.nunique())

Training Set: (11563,) (11563,)
Validation Set: (2478,) (2478,)
Test Set: (2478,) (2478,)

Unique categories:
Training: 12
Validation: 12
Test: 12


In [6]:
# Compare category proportions across the three subsets
split_distribution = pd.DataFrame({
    "Original": y.value_counts(normalize=True) * 100,
    "Training": y_train.value_counts(normalize=True) * 100,
    "Validation": y_val.value_counts(normalize=True) * 100,
    "Test": y_test.value_counts(normalize=True) * 100
})

# Display the percentage distribution for each category
display(split_distribution.round(2))

,Original,Training,Validation,Test
target_category,,,,
Road Infrastructure,31.62,31.62,31.64,31.64
Waste Management,25.64,25.64,25.63,25.63
Street Lighting,8.91,8.91,8.92,8.92
Traffic and Transport,7.02,7.01,7.02,7.02
Drainage and Sewerage,6.09,6.09,6.09,6.09
Animal Management,5.25,5.26,5.25,5.25
Environment and Public Spaces,4.59,4.58,4.60,4.60
Water Supply,3.64,3.65,3.63,3.63
Public Sanitation,2.91,2.91,2.91,2.91


# 4. Build and Fit the Tokenizer

This section creates a Keras tokenizer and fits it exclusively
on the training complaint texts.

An out-of-vocabulary token is included to handle words that are
not present in the tokenizer vocabulary.

The same fitted tokenizer will be used for validation, testing
and future prediction data.# 4. Build and Fit the Tokenizer

This section creates a Keras tokenizer and fits it exclusively
on the training complaint texts.

An out-of-vocabulary token is included to handle words that are
not present in the tokenizer vocabulary.

The same fitted tokenizer will be used for validation, testing
and future prediction data.

In [7]:
# Import the Keras tokenizer for converting text into integer sequences
from tensorflow.keras.preprocessing.text import Tokenizer

# Import the padding utility for later sequence preparation
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [8]:
# Initialize the tokenizer with an unknown-word token
tokenizer = Tokenizer(oov_token="<OOV>")

# Fit the tokenizer only on training complaint texts
tokenizer.fit_on_texts(X_train)

# Retrieve the word-to-integer mapping
word_index = tokenizer.word_index

# Display the total number of unique tokens
print("Vocabulary Size:", len(word_index))

# Display the first 20 word-to-integer mappings
print("\nFirst 20 Vocabulary Entries:")
print(list(word_index.items())[:20])

# Check the integer assigned to the OOV token
print("\nOOV Token Index:", word_index["<OOV>"])

Vocabulary Size: 20088

First 20 Vocabulary Entries:
[('<OOV>', 1), ('the', 2), ('and', 3), ('to', 4), ('road', 5), ('is', 6), ('of', 7), ('in', 8), ('this', 9), ('a', 10), ('on', 11), ('garbage', 12), ('are', 13), ('not', 14), ('for', 15), ('it', 16), ('no', 17), ('from', 18), ('there', 19), ('please', 20)]

OOV Token Index: 1


In [9]:
# Convert training complaints into sequences of integer token IDs
train_sequences = tokenizer.texts_to_sequences(X_train)

# Calculate the number of tokens in each complaint
sequence_lengths = [len(sequence) for sequence in train_sequences]

# Display descriptive statistics of sequence lengths
print("Sequence Length Statistics:")
print(pd.Series(sequence_lengths).describe(percentiles=[0.90, 0.95, 0.98, 0.99]))

# Display selected percentiles to help choose the padding length
for percentile in [90, 95, 98, 99]:
    print(f"{percentile}th Percentile:", np.percentile(sequence_lengths, percentile))

Sequence Length Statistics:
count    11563.000000
mean        46.577013
std         39.550456
min          0.000000
90%         92.000000
95%        118.000000
98%        156.000000
99%        195.000000
max        747.000000
dtype: float64
90th Percentile: 92.0
95th Percentile: 118.0
98th Percentile: 156.0
99th Percentile: 195.0


In [10]:
# Set the maximum sequence length using the 98th percentile
MAX_LEN = 156

# Convert training complaints into integer sequences
train_sequences = tokenizer.texts_to_sequences(X_train)

# Convert validation complaints using the same fitted tokenizer
val_sequences = tokenizer.texts_to_sequences(X_val)

# Convert test complaints using the same fitted tokenizer
test_sequences = tokenizer.texts_to_sequences(X_test)

# Pad shorter sequences and truncate longer sequences
X_train_padded = pad_sequences(
    train_sequences, maxlen=MAX_LEN, padding="post", truncating="post"
)

# Apply identical padding and truncation to validation data
X_val_padded = pad_sequences(
    val_sequences, maxlen=MAX_LEN, padding="post", truncating="post"
)

# Apply identical padding and truncation to test data
X_test_padded = pad_sequences(
    test_sequences, maxlen=MAX_LEN, padding="post", truncating="post"
)

# Display the resulting input shapes
print("Training Shape:", X_train_padded.shape)
print("Validation Shape:", X_val_padded.shape)
print("Testing Shape:", X_test_padded.shape)

Training Shape: (11563, 156)
Validation Shape: (2478, 156)
Testing Shape: (2478, 156)


In [11]:
# Import LabelEncoder for converting category names into numerical labels
from sklearn.preprocessing import LabelEncoder

# Create the label encoder
label_encoder = LabelEncoder()

# Fit the encoder on training labels and transform them
y_train_encoded = label_encoder.fit_transform(y_train)

# Transform validation and test labels using the same encoder
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

# Display the mapping between category names and numerical labels
label_mapping = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))

print("Label Mapping:")
for category, label in label_mapping.items():
    print(f"{category}: {label}")

# Display the shapes of encoded labels
print("\nEncoded Label Shapes:")
print("Training:", y_train_encoded.shape)
print("Validation:", y_val_encoded.shape)
print("Testing:", y_test_encoded.shape)

# Verify that all 12 categories are represented
print("\nNumber of Classes:", len(label_encoder.classes_))

Label Mapping:
Animal Management: 0
Civic Infrastructure and Encroachment: 1
Drainage and Sewerage: 2
Electricity and Utilities: 3
Environment and Public Spaces: 4
Public Safety: 5
Public Sanitation: 6
Road Infrastructure: 7
Street Lighting: 8
Traffic and Transport: 9
Waste Management: 10
Water Supply: 11

Encoded Label Shapes:
Training: (11563,)
Validation: (2478,)
Testing: (2478,)

Number of Classes: 12


In [12]:
# Import libraries for saving preprocessing objects and arrays
import os
import pickle
import numpy as np

# Create a directory for storing tokenization artifacts
os.makedirs("../models/preprocessing", exist_ok=True)

# Save the fitted tokenizer for reuse in all model notebooks
with open("../models/preprocessing/tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)

# Save the fitted label encoder to preserve the category mapping
with open("../models/preprocessing/label_encoder.pkl", "wb") as file:
    pickle.dump(label_encoder, file)

# Save the maximum sequence length used for padding
np.save("../models/preprocessing/max_len.npy", MAX_LEN)

# Save the padded training, validation and test sequences
np.savez_compressed(
    "../models/preprocessing/tokenized_data.npz",
    X_train=X_train_padded,
    X_val=X_val_padded,
    X_test=X_test_padded,
    y_train=y_train_encoded,
    y_val=y_val_encoded,
    y_test=y_test_encoded
)

# Confirm that the preprocessing artifacts have been saved
print("Tokenizer, label encoder, maximum length and tokenized datasets saved successfully.")

Tokenizer, label encoder, maximum length and tokenized datasets saved successfully.
